### Thierry (Assister par IA)
## Carte de clustering

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn folium scikit-learn joblib

## Imports et configuration

Import des bibliothèques nécessaires :
- `pandas` / `numpy` : manipulation des données
- `scikit-learn` : algorithme `KMeans` et métriques d'évaluation du clustering
- `folium` : cartes interactives (OpenStreetMap)
- `joblib` : sérialisation du modèle entraîné
- `matplotlib` / `seaborn` : visualisations statiques
- `os` : gestion des fichiers

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
import folium
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("output", exist_ok=True)

## Chargement des données

Lecture du fichier CSV exporté depuis le registre national des bornes de recharge (IRVE).
Le dataset contient ~138 000 points de charge avec 47 colonnes (opérateur, localisation, puissance, horaires, etc.).

In [ ]:
# Chargement des données IRVE
irve = pd.read_csv("export_IA.csv")

irve.head()

## Apprentissage non-supervisé

**Variables retenues :** `latitude`, `longitude`.

**Justification du choix des variables :**
- Seules `latitude` / `longitude` sont utilisées : le besoin est une sectorisation purement géographique des bornes, sans tenir compte de leurs autres caractéristiques (puissance, opérateur, type d'implantation…).
- `dropna` est appliqué sur ces deux colonnes uniquement — une borne sans coordonnées ne peut pas être positionnée sur la carte ni assignée à un cluster.

In [ ]:
# Extraction de la longitude et latitude
irve = irve.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
X = irve[["latitude", "longitude"]]

## Recherche de nombre de cluster

**Justification du choix des métriques :**
- **Silhouette Score** : mesure la cohésion intra-cluster vs la séparation inter-cluster, idéal le plus HAUT. Calculé sur un échantillon de 5000 points (`sample_size=5000`) pour rester rapide malgré les ~139 000 bornes.
- **Calinski-Harabasz** : ratio de dispersion inter/intra-cluster, idéal le plus HAUT. Sensible aux clusters bien séparés, complémentaire au Silhouette.
- **Davies-Bouldin** : mesure la similarité moyenne entre clusters voisins, idéal le plus BAS. Permet de détecter des clusters trop proches les uns des autres.

Les valeurs de K testées (2, 3, 4, 5, 6, 7, 12) couvrent une plage large pour situer visuellement où se trouve le meilleur compromis avant de restreindre le choix final à 5, 6 ou 7 (cf. section suivante).

In [ ]:
results = []
range_n_clusters = [2, 3, 4, 5, 6, 7,12]

for k in range_n_clusters:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)

    # Calcul des métriques demandées
    sil = silhouette_score(X, cluster_labels, sample_size=5000, random_state=42)
    ch = calinski_harabasz_score(X, cluster_labels)
    db = davies_bouldin_score(X, cluster_labels)

    results.append(
        {"K": k, "Silhouette": sil, "Calinski-Harabasz": ch, "Davies-Bouldin": db}
    )

## Affichage du tableau comparatif pour votre rapport

In [ ]:
# Affichage du tableau comparatif pour votre rapport
df_metrics = pd.DataFrame(results)
print("\n--- TABLEAU DES MÉTRIQUES D'ÉVALUATION ---")
print(df_metrics.to_string(index=False))
print("------------------------------------------")
# 2. Configuration de l'affichage (1 ligne, 3 graphiques)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graphique 1 : Score de Silhouette
sns.lineplot(ax=axes[0], data=df_metrics, x='K', y='Silhouette', marker='o', linewidth=2, color='#d95f02')
axes[0].set_title("Score de Silhouette\n(Idéal : le plus HAUT)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Nombre de clusters (K)")
axes[0].set_ylabel("Silhouette Score")
axes[0].grid(True, linestyle='--', alpha=0.6)

# Graphique 2 : Indice Calinski-Harabasz
sns.lineplot(ax=axes[1], data=df_metrics, x='K', y='Calinski-Harabasz', marker='o', linewidth=2, color='#7570b3')
axes[1].set_title("Indice Calinski-Harabasz\n(Idéal : le plus HAUT)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Nombre de clusters (K)")
axes[1].set_ylabel("CH Score")
axes[1].grid(True, linestyle='--', alpha=0.6)

# Graphique 3 : Indice Davies-Bouldin
sns.lineplot(ax=axes[2], data=df_metrics, x='K', y='Davies-Bouldin', marker='o', linewidth=2, color='#1b9e77')
axes[2].set_title("Indice Davies-Bouldin\n(Idéal : le plus BAS)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Nombre de clusters (K)")
axes[2].set_ylabel("DB Score")
axes[2].grid(True, linestyle='--', alpha=0.6)

# Ajustement automatique des espaces et affichage
plt.tight_layout()
plt.savefig(os.path.join("output", "metriques_clustering.png"))
plt.show()

## Choix du nombre de clusters et entraînement du modèle final

**Justification du choix de K :**
- Le besoin client impose de restreindre le choix à 5, 6 ou 7 clusters — un compromis entre une sectorisation trop grossière (moins de 5 zones) et trop fragmentée (plus de 7 zones) pour rester opérationnelle.
- `k_choisi` ci-dessous est la valeur retenue (modifiable manuellement) ; le tableau de métriques de la section précédente permet de comparer 5, 6 et 7 avant de trancher.
- Une fois `k_choisi` fixé, le modèle `KMeans` est entraîné sur l'ensemble des données puis sauvegardé (`joblib`) pour être réutilisé sans réentraînement par `script.py`.

In [ ]:
# Nombre de clusters choisi par l'utilisateur : modifier cette valeur (5, 6 ou 7)
k_choisi = 5

if k_choisi not in (5, 6, 7):
    raise ValueError("k_choisi doit être 5, 6 ou 7.")

print(f"Nombre de clusters choisi : {k_choisi}")
print(f"Entraînement du modèle final avec K={k_choisi}...")
final_kmeans = KMeans(n_clusters=k_choisi, random_state=42, n_init=10)
final_kmeans.fit(X)

# SAUVEGARDE STRICTE DU MODÈLE
model_filename = "kmeans_irve_model.pkl"
joblib.dump(final_kmeans, model_filename)
print(f"Modèle sauvegardé avec succès sous le nom : '{model_filename}'")

## Creation de la carte de clustering

**Justification du choix de Folium + FeatureGroup (sans MarkerCluster) :**
- **Folium** : bibliothèque Python légère basée sur Leaflet.js. Produit des cartes HTML interactives sans serveur — idéal pour partager des résultats en standalone.
- **CartoDB Positron** : tuiles utilisées à la place d'OpenStreetMap pour éviter les erreurs réseau (403 Forbidden) et offrir une meilleure lisibilité des clusters.
- **FeatureGroup par cluster, sans `MarkerCluster`** : contrairement à la carte du Besoin Client 1, l'objectif ici n'est pas de regrouper visuellement les bornes proches mais de garder une couleur distincte et lisible pour chaque cluster K-Means. Un `MarkerCluster` mélangerait les couleurs et masquerait les frontières entre clusters.

La carte est centrée sur la moyenne des positions des bornes. Un contrôle de couches permet d'afficher/masquer chaque cluster indépendamment.

In [ ]:
# Rechargement du modèle préalablement enregistré
final_kmeans = joblib.load("kmeans_irve_model.pkl")

# Assignation des clusters aux données existantes
X = irve[["latitude", "longitude"]]
irve["cluster"] = final_kmeans.predict(X)

# Création de la carte centrée sur la moyenne des positions
map_center = [irve["latitude"].mean(), irve["longitude"].mean()]
carte = folium.Map(location=map_center, zoom_start=6, tiles="CartoDB positron", prefer_canvas=True)

# Palette de couleurs pour différencier les clusters
colors = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue"]

# Un FeatureGroup par cluster (sans MarkerCluster) : chaque borne garde sa
# couleur de cluster visible individuellement, contrairement à un
# regroupement par proximité géographique (qui mélangerait les couleurs et
# masquerait les frontières entre clusters).
cluster_groups = {}
for cluster_id in sorted(irve["cluster"].unique()):
    cluster_groups[cluster_id] = folium.FeatureGroup(name=f"Cluster {cluster_id}").add_to(carte)

for idx, row in irve.iterrows():
    cluster_id = int(row["cluster"])
    color = colors[cluster_id % len(colors)]

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=f"Borne ID: {idx}<br>Cluster: {cluster_id}",
    ).add_to(cluster_groups[cluster_id])

folium.LayerControl(collapsed=False).add_to(carte)

In [ ]:
chemin_carte = os.path.join("output", "carte_clusters_irve.html")
carte.save(chemin_carte)
print(
    f"La carte interactive a été générée sous le nom '{chemin_carte}'. Ouvrez-la dans un navigateur !"
)